# Demo — migration sanity check

This notebook does **not** reproduce all 24 datasets. Its only purpose is to
prove that the package migration works end-to-end:

1. Load `config/experiment.yaml`.
2. Build a `DataLoader` for the **baseline** dataset.
3. Train one method (**naive**) for the configured number of tasks.
4. Print the thesis metrics.

To reproduce the full study, use the CLI (Command Line Interface) runner:

```bash
python -m scripts.run_experiments --config config/experiment.yaml
```


## 1. Setup — make `src/` importable and load the config

In [ ]:
import sys
from pathlib import Path

# Notebook lives in <project>/notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
with open(PROJECT_ROOT / "config" / "experiment.yaml") as f:
    cfg = yaml.safe_load(f)

print("Project root:", PROJECT_ROOT)
print("Loaded config keys:", list(cfg.keys()))


## 2. Pick the baseline dataset

In [ ]:
from src.data import DataLoader

base_dir = Path(cfg["paths"]["base_dir"])
if not base_dir.is_absolute():
    base_dir = PROJECT_ROOT / base_dir

dataset_csv = base_dir / "baseline_pattern_A_stationary.csv"
print("Dataset path:", dataset_csv)
print("Exists?     ", dataset_csv.exists())

loader = DataLoader(dataset_csv)


## 3. Train the **naive** baseline for `n_tasks` tasks

In [ ]:
import numpy as np
from src.methods import FraudDetector, train_on_task
from src.utils import compute_thesis_metrics, set_seed, test_on_task

set_seed(cfg["seed"])

n_tasks = cfg["n_tasks"]
model = FraudDetector(input_features=cfg["n_features"])
acc = np.zeros((n_tasks, n_tasks))
pr  = np.zeros((n_tasks, n_tasks))

for task_id in range(n_tasks):
    X_train, y_train = loader.get_data_for_task(task_id, "train")
    model = train_on_task(
        model, X_train, y_train,
        epochs=cfg["n_epochs"], lr=cfg["learning_rate"],
    )
    for test_id in range(task_id + 1):
        X_test, y_test = loader.get_data_for_task(test_id, "test")
        m = test_on_task(model, X_test, y_test)
        acc[task_id, test_id] = m["acc"]
        pr[task_id,  test_id] = m["pr_auc"]

print("Training done.")


## 4. Aggregate metrics

In [ ]:
avg_acc, forget_acc, avg_pr, forget_pr, diag_pr = compute_thesis_metrics(acc, pr)

print(f"AvgAcc     = {avg_acc:.4f}")
print(f"ForgetAcc  = {forget_acc:.4f}")
print(f"AvgPR      = {avg_pr:.4f}")
print(f"ForgetPR   = {forget_pr:.4f}")
print(f"DiagPR     = {diag_pr:.4f}")


## 5. (Optional) Persist the result to disk

If the dataset CSV is present, the snippet below saves the matrices to
`res/experiment_results/baseline/naive_*.npy`, exactly like the CLI runner
would. Skip this cell if you only want a smoke test.


In [ ]:
# from src.utils import save_ckpt

# results_dir = Path(cfg["paths"]["results_dir"])
# if not results_dir.is_absolute():
#     results_dir = PROJECT_ROOT / results_dir

# save_ckpt(results_dir, ds_name="baseline", method="naive", acc_m=acc, pr_m=pr)
# print("Saved to", results_dir / "baseline")
